# Top-level imports and mount to google drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Access our shared folder so we can use our downloaded data

In [ ]:
# Define the path to specific folder in MyDrive
specific_folder_path = '/content/drive/MyDrive/DD2424_project_2026'

# Check if the folder exists and list its contents
if os.path.exists(specific_folder_path):
    print(f"Contents of '{specific_folder_path}':")
    for item in os.listdir(specific_folder_path):
        print(item)
else:
    print(f"The folder '{specific_folder_path}' does not exist. Please check the path.")

### Untarring `annotations.tar` and `images.tar`

We will extract the contents of the `annotations.tar` and `images.tar` files. 

In [ ]:
import os

# Define the path to the Data folder within project directory
project_data_path = os.path.join(specific_folder_path, 'Data')

# Define the base directory where extracted files will be stored
extracted_base_path = '/content/DD2424_project_2026_extracted'

# Create subdirectories for annotations and images
annotations_output_path = os.path.join(extracted_base_path, 'annotations')
images_output_path = os.path.join(extracted_base_path, 'images')

os.makedirs(annotations_output_path, exist_ok=True)
os.makedirs(images_output_path, exist_ok=True)

print(f"Created extraction directory: {annotations_output_path}")
print(f"Created extraction directory: {images_output_path}")


In [ ]:
# Path to the annotations tar file
annotations_tar_file = os.path.join(project_data_path, 'annotations.tar')

# Untar annotations.tar into the 'annotations' subdirectory
print(f"Extracting {annotations_tar_file} to {annotations_output_path}...")
!tar -xf {annotations_tar_file} -C {annotations_output_path}
print("Extraction complete for annotations.tar")

# List contents to verify
print(f"Contents of {annotations_output_path}:")
!ls {annotations_output_path}

In [ ]:
# Path to the images tar file
images_tar_file = os.path.join(project_data_path, 'images.tar')

# Untar images.tar into the 'images' subdirectory
print(f"Extracting {images_tar_file} to {images_output_path}...")
!tar -xf {images_tar_file} -C {images_output_path}
print("Extraction complete for images.tar")

# List contents to verify
print(f"Contents of {images_output_path}:")
!ls {images_output_path}

# Reading one image from our new local directory

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# The actual images seem to be in a nested 'images' directory
actual_images_path = os.path.join(images_output_path, 'images')

# List all files in the images directory to find an image to plot
image_files = [f for f in os.listdir(actual_images_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if image_files:
    # Pick the first image found
    sample_image_name = image_files[0]
    sample_image_path = os.path.join(actual_images_path, sample_image_name)

    print(f"Plotting sample image: {sample_image_name}")

    # Load the image
    img = mpimg.imread(sample_image_path)

    # Display the image
    plt.imshow(img)
    plt.title(f"Sample Image: {sample_image_name}")
    plt.axis('off') # Hide axes ticks and labels
    plt.show()
else:
    print(f"No image files (png, jpg, jpeg) found in {actual_images_path}.")

# Access, and split the data up for binary classification dog vs cat


Excerpt from the '/annotations/annotations/README.txt' file:


```
We have created a 37 category pet dataset with roughly 200 images for each class.
The images have a large variations in scale, pose and lighting. All images have an
associated ground truth annotation of breed, head ROI, and pixel
level trimap segmentation.

Contents:
--------
trimaps/ 	Trimap annotations for every image in the dataset
		Pixel Annotations: 1: Foreground 2:Background 3: Not classified
xmls/		Head bounding box annotations in PASCAL VOC Format

list.txt	Combined list of all images in the dataset
		Each entry in the file is of following nature:
		Image CLASS-ID SPECIES BREED ID
		ID: 1:37 Class ids
		SPECIES: 1:Cat 2:Dog
		BREED ID: 1-25:Cat 1:12:Dog
		All images with 1st letter as captial are cat images while
		images with small first letter are dog images.
    trainval.txt	Files describing splits used in the paper.However,
    test.txt	you are encouraged to try random splits.
```





In [ ]:
def get_breeds_for_multiclass():
    breeds = {} # key: file prefix, val: breed id
    with open(f"{annotations_output_path}/annotations/list.txt", "r") as f:
        for line in f:
            # Skip comments or empty lines
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            image_name = parts[0]
            breed_id = parts[3]
            breeds[image_name] = breed_id
    return breeds
print("Mapping cat and dog breeds...")
breeds = get_breeds_for_multiclass()

In [ ]:
file_names = os.listdir("/content/DD2424_project_2026_extracted/images/images/")
path = "/content/DD2424_project_2026_extracted/images/images/"

In [ ]:
# Re-generating the labels dictionary to fix the TypeError
labels = {}
for file_name in file_names:
    split_name = file_name.split('.')
    base_name, extension = split_name[0], split_name[1]
    if extension == 'mat': # skip matlab files
        continue
    if base_name in breeds.keys():
        labels[file_name] = breeds[base_name]

# Create a list of filenames that actually have labels
labeled_filenames = list(labels.keys())

print(f'Number of labeled images: {len(labeled_filenames)}')
print(f'Sample labels keys: {list(labels.keys())[:5]}')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class PetDataset(Dataset):
    def __init__(self, image_dir, filenames, labels_dict, transform=None):
        self.image_dir = image_dir
        self.filenames = list(filenames)
        self.labels_dict = labels_dict
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        img_path = os.path.join(self.image_dir, img_name)

        # Load image
        image = Image.open(img_path).convert('RGB')

        # Get label (1-37 from list.txt, map to 0-36 for PyTorch)
        label = int(self.labels_dict[img_name]) - 1

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
# FOR THE LIMITED DATA EXPERIMENTS
# Define the Augmented Transform (Requirement: flip, rotation, crops)
train_transform_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_stratified_subset(filenames, labels_dict, fraction):
    """Returns a subset of filenames stratified by class to keep data balanced."""
    if fraction >= 1.0:
        return filenames
    labels_list = [int(labels_dict[f]) for f in filenames]

    subset_filenames, _ = train_test_split(
        filenames,
        train_size=fraction,
        stratify=labels_list,
        random_state=42
    )
    return subset_filenames

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def imshow(tensor, title=None):
    # Convert from Tensor to Numpy
    image = tensor.cpu().numpy().transpose((1, 2, 0))

    # Undo the ImageNet Normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean

    # Clip to stay between 0 and 1 (standard for matplotlib)
    image = np.clip(image, 0, 1)

    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

# Process the images: Ensure they are the same size, etc.

We use our torch dataset class along with dataloader to follow torch convention, also so that we can load our data in batches in a way that doesn't crash our gpu. Dataset class is used to standardize how we fetch individual samples from our directory along with an associated label. DataLoader wraps this class and returns batches of torch images along with their labels!

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image # Pillow is used by torchvision to load images
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.model_selection import train_test_split


# Define transformations (STANDARD FOR CIFAR)
transform_pipeline = transforms.Compose([
      transforms.Resize((224, 224)), # Resize image to 224x224 pixels (common input size for CNNs)
      transforms.ToTensor(),        # Convert PIL Image or numpy.ndarray to a PyTorch Tensor
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with ImageNet mean/std
  ])

# Split data into training, validation, and test sets
train_filenames, test_filenames = train_test_split(labeled_filenames, test_size=0.15, random_state=42)
train_filenames, val_filenames = train_test_split(train_filenames, test_size=(0.15/0.85), random_state=42) # Split remaining 85% into train and validation

print(f"Number of training images: {len(train_filenames)}")
print(f"Number of validation images: {len(val_filenames)}")
print(f"Number of test images: {len(test_filenames)}")

# Create Dataset instances for each split
train_dataset = PetDataset(image_dir=path, filenames=train_filenames, labels_dict=labels, transform=transform_pipeline)
val_dataset = PetDataset(image_dir=path, filenames=val_filenames, labels_dict=labels, transform=transform_pipeline)
test_dataset = PetDataset(image_dir=path, filenames=test_filenames, labels_dict=labels, transform=transform_pipeline)

# Create DataLoader instances for each split
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) # No need to shuffle validation/test sets
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Display a sample image from the training loader
print("Displaying a sample image from the training set:")
for images, batch_labels in train_loader:
    imshow(images[0])
    break

# Training with a resnet 18

In [ ]:
import torch.nn as nn
from torchvision import models

# Load the pre-trained ResNet18
model = models.resnet18(weights='DEFAULT')

# Freeze all params in model

In [ ]:
for param in model.parameters():
    param.requires_grad = False # We're telling autograd not to track and compute gradients for these weights

# Unfreeze linear layer

In [ ]:
# Check how many inputs the last layer had
num_ftrs = model.fc.in_features

# Replace the last layer, is automatically unfrozen
model.fc = nn.Linear(num_ftrs, 37)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set optimizer - ADAM

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

# Only optimize the parameters of the final layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# TRAIN THE MODEL!!!

In [ ]:
from tqdm import tqdm
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(10):
    running_loss = 0.0
    running_corrects = 0

    model.train()  # training mode

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/10"):

        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # reset optimizer
        output = model(images)  # forward pass
        _, preds = torch.max(output, 1)

        loss = criterion(output, labels)  # get loss
        loss.backward()  # backprop
        optimizer.step()  # alter weights

        running_loss += loss.item() * images.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = running_corrects.double() / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    model.eval()
    running_val_loss = 0.0
    running_correct_val = 0

    with torch.no_grad():
        for val_image, val_label in tqdm(val_loader, "Validation mode"):
            val_image, val_label = val_image.to(device), val_label.to(device)
            out_val = model(val_image)
            loss = criterion(out_val, val_label)
            _, preds_val = torch.max(out_val, 1)
            running_val_loss += loss.item() * val_image.size(0)
            running_correct_val += torch.sum(preds_val == val_label.data)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_val_acc = running_correct_val.double() / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    print(f"Validation Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        print(f"Validation loss decreased. Saving best model to 'best_model.pth'...")
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss
        }
        torch.save(checkpoint, 'best_model_resnet_18.pth')

In [ ]:
import torch
model = model.to(device)
checkpoint_path = 'best_model_resnet_18.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Successfully loaded model from Epoch {checkpoint['epoch']} (Val Loss: {checkpoint['best_val_loss']:.4f})")
model.eval()
test_loss = 0.0
test_corrects = 0
with torch.no_grad():
    for test_images, test_labels in tqdm(test_loader, desc="Testing Mode"):
        test_images, test_labels = test_images.to(device), test_labels.to(device)
        outputs = model(test_images)
        loss = criterion(outputs, test_labels)
        _, preds = torch.max(outputs, 1)
        test_loss += loss.item() * test_images.size(0)
        test_corrects += torch.sum(preds == test_labels.data)
final_test_loss = test_loss / len(test_loader.dataset)
final_test_acc = test_corrects.double() / len(test_loader.dataset)
print("\nFINAL TEST RESULTS")
print(f"Test Loss: {final_test_loss:.4f} | Test Accuracy: {final_test_acc:.4f}")

## Start making this more scalable for more expeeriments

In [ ]:
import torch
from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10, save_path='best_model.pth'):
    train_losses, val_losses = [], []
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        # TRAINING 
        running_loss = 0.0
        running_corrects = 0
        model.train()
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            _, preds = torch.max(output, 1)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # VALIDATION 
        model.eval()
        running_val_loss = 0.0
        running_correct_val = 0
        with torch.no_grad():
            for val_image, val_label in tqdm(val_loader, desc="Validation mode"):
                val_image, val_label = val_image.to(device), val_label.to(device)
                out_val = model(val_image)
                loss = criterion(out_val, val_label)
                _, preds_val = torch.max(out_val, 1)
                running_val_loss += loss.item() * val_image.size(0)
                running_correct_val += torch.sum(preds_val == val_label.data)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = running_correct_val.double() / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        print(f"Validation Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # CHECKPOINTING
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            print(f"Validation loss decreased. Saving best model to '{save_path}'...")
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss
            }
            torch.save(checkpoint, save_path)

    # Load from the exact same save_path parameter
    print(f"Training complete. Loading best model weights from {save_path}")
    model.load_state_dict(torch.load(save_path)['model_state_dict'])

    return model, train_losses, val_losses

In [ ]:
def test_model(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0.0
    test_corrects = 0

    with torch.no_grad():
        for test_images, test_labels in tqdm(test_loader, desc="Testing Mode"):
            test_images, test_labels = test_images.to(device), test_labels.to(device)

            outputs = model(test_images)
            loss = criterion(outputs, test_labels)
            _, preds = torch.max(outputs, 1)

            test_loss += loss.item() * test_images.size(0)
            test_corrects += torch.sum(preds == test_labels.data)

    final_test_loss = test_loss / len(test_loader.dataset)
    final_test_acc = test_corrects.double() / len(test_loader.dataset)

    print("\n" + "="*30)
    print("FINAL TEST RESULTS")
    print(f"Test Loss: {final_test_loss:.4f}")
    print(f"Test Accuracy: {final_test_acc:.4f}")
    print("="*30 + "\n")

    return final_test_loss, final_test_acc.item()

# Strat 1: Unfreeze L layers simultaneously



In [ ]:
import torchvision.models as models
import torch.nn as nn

def get_resnet_strategy_1(l=1, num_classes=37):
    model = models.resnet18(weights='DEFAULT')
    for param in model.parameters():
        param.requires_grad = False
    blocks = [model.layer4, model.layer3, model.layer2, model.layer1]
    if l > 0:
        l = min(l, len(blocks))
        for i in range(l):
            for param in blocks[i].parameters():
                param.requires_grad = True
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    unfrozen_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Created ResNet18 with L={l} unfrozen blocks. Trainable parameters: {unfrozen_params:,}")
    return model

In [ ]:
# STRAT 1
l_values_to_test = [1, 2, 3, 4]
strategy1_results = {}

for l in l_values_to_test:
    print(f"\n" + "="*40)
    print(f"STARTING STRATEGY 1: L = {l}")
    print("="*40)
    model = get_resnet_strategy_1(l=l, num_classes=37).to(device)

    criterion = nn.CrossEntropyLoss()
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params_to_update, lr=0.001)
    save_name = f'strategy1_L{l}_best.pth'
    best_model, train_loss, val_loss = train_model(
        model, train_loader, val_loader, criterion, optimizer, device,
        num_epochs=10, save_path=save_name
    )
    test_loss, test_acc = test_model(best_model, test_loader, criterion, device)
    strategy1_results[f"L={l}"] = val_loss

In [ ]:
def unfreeze_resnet_block(model, block_name):
    for name, module in model.named_children():
        if name == block_name:
            for param in module.parameters():
                param.requires_grad = True
            print(f"Successfully unfrozen: {block_name}")
            return
    print(f"Warning: Could not find block {block_name}")

In [ ]:
# Strat 2 (GRADUAL UNFREEZING) 
print("="*50)
print("STARTING STRATEGY 2: GRADUAL UNFREEZING")
print("="*50)

model_strat2 = get_resnet_strategy_1(l=0, num_classes=37).to(device) # l = 0
criterion = nn.CrossEntropyLoss()
strat2_train_loss_history = []
strat2_val_loss_history = []

# Define training stages
stages = [
    {'stage_name': 'Stage 1 (FC Only)',  'unfreeze': None,     'lr': 1e-3, 'epochs': 5},
    {'stage_name': 'Stage 2 (+ Layer 4)', 'unfreeze': 'layer4', 'lr': 1e-4, 'epochs': 5},
    {'stage_name': 'Stage 3 (+ Layer 3)', 'unfreeze': 'layer3', 'lr': 1e-5, 'epochs': 5}
]

for stage in stages:
    print(f"\n--- {stage['stage_name']} ---")
    # Unfreeze
    if stage['unfreeze']:
        unfreeze_resnet_block(model_strat2, stage['unfreeze'])

    # Create a new optimizer for this stage.
    # Why? Because we have newly unfrozen parameters, and we are lowering the learning rate!
    params_to_update = [p for p in model_strat2.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params_to_update, lr=stage['lr'])

    # Train the model for this stage
    save_name = f"strategy2_best.pth" # Overwrites with the absolute best across all stages
    model_strat2, t_loss, v_loss = train_model(
        model_strat2, train_loader, val_loader, criterion, optimizer, device,
        num_epochs=stage['epochs'], save_path=save_name
    )
    strat2_train_loss_history.extend(t_loss)
    strat2_val_loss_history.extend(v_loss)

# Finally, test the absolute best model from this gradual unfreezing process
print("\nEvaluating final Gradual Unfreezing model on Test Set...")
test_loss, test_acc = test_model(model_strat2, test_loader, criterion, device)

# Limited data

In [ ]:
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# LIMITED DATA EXPERIMENTS GRID 
data_fractions = [1, 0.1, 0.01]
use_augmentation_options = [False, True]
# 0.0 means no L2 regularization
weight_decay_options = [0.0, 1e-5, 1e-4, 1e-3]

results_log = []

for frac in data_fractions:
    # Get the limited stratified filenames
    subset_train_files = get_stratified_subset(train_filenames, labels, frac)

    for use_aug in use_augmentation_options:
        # Choose the transform: standard vs. augmented
        current_transform = train_transform_aug if use_aug else transform_pipeline

        # Prepare the data loader for this specific limited-data run
        limited_train_ds = PetDataset(
            path, subset_train_files, labels, current_transform)
        limited_train_loader = DataLoader(
            limited_train_ds, batch_size=32, shuffle=True)

        for wd in weight_decay_options:
            # Iterate through both Transfer Learning Strategies
            for strat_name in ['Strategy 1 (L=1)', 'Strategy 2 (Gradual)']:
                print(f"\n" + "="*60)
                print(
                    f"RUNNING: {int(frac*100)}% Data | {strat_name} | Augmentation: {use_aug} | Weight Decay: {wd}")
                print("="*60)

                if strat_name == 'Strategy 1 (L=1)':

                    model = get_resnet_strategy_1(
                        l=1, num_classes=37).to(device)

                    # Use Adam with L2 Regularization 
                    optimizer = torch.optim.Adam(
                        filter(lambda p: p.requires_grad, model.parameters()),
                        lr=0.001,
                        weight_decay=wd 
                    )

                    trained_model, _, _ = train_model(
                        model, limited_train_loader, val_loader,
                        nn.CrossEntropyLoss(), optimizer, device,
                        num_epochs=10,
                        save_path=f'best_s1_f{frac}_aug{use_aug}_wd{wd}.pth'
                    )

                else:

                    # Initialize model with only FC layer unfrozen
                    model_strat2 = get_resnet_strategy_1(
                        l=0, num_classes=37).to(device)
                    criterion = nn.CrossEntropyLoss()

                    # Define training stages for gradual unfreezing
                    stages = [
                        {'stage_name': 'Stage 1 (FC Only)',  'unfreeze': None,
                         'lr': 1e-3, 'epochs': 5},
                        {'stage_name': 'Stage 2 (+ Layer 4)',
                         'unfreeze': 'layer4', 'lr': 1e-4, 'epochs': 5},
                        {'stage_name': 'Stage 3 (+ Layer 3)',
                         'unfreeze': 'layer3', 'lr': 1e-5, 'epochs': 5}
                    ]

                    for stage in stages:
                        print(f"\n--- {strat_name}: {stage['stage_name']} ---")
                        if stage['unfreeze']:
                            unfreeze_resnet_block(
                                model_strat2, stage['unfreeze'])

                        # Reinitialize optimizer with newly unfrozen params and lower learning rate
                        params_to_update = [
                            p for p in model_strat2.parameters() if p.requires_grad]
                        # Use current weight_decay
                        optimizer = torch.optim.Adam(
                            params_to_update, lr=stage['lr'], weight_decay=wd)

                        # Train using the current stage configuration
                        model_strat2, _, _ = train_model(
                            model_strat2, limited_train_loader, val_loader,
                            criterion, optimizer, device,
                            num_epochs=stage['epochs'],
                            save_path=f"best_s2_f{frac}_aug{use_aug}_wd{wd}.pth"
                        )
                    trained_model = model_strat2

                # Final Evaluation on the unseen Test Set
                print(
                    f"\nFinal evaluation for {strat_name} ({int(frac*100)}% data, Aug: {use_aug}, WD: {wd})")
                t_loss, t_acc = test_model(
                    trained_model, test_loader, nn.CrossEntropyLoss(), device)

                # Store results
                results_log.append({
                    "fraction": frac,
                    "strategy": strat_name,
                    "augmentation": use_aug,
                    "weight_decay": wd,
                    "test_acc": t_acc,
                    "test_loss": t_loss
                })

# Save results for report
with open('limited_data_results.json', 'w') as f:
    json.dump(results_log, f, indent=4)

print("\n" + "="*60)
print("ALL LIMITED DATA EXPERIMENTS COMPLETE.")
print("Results saved to 'limited_data_results.json' for analysis.")
print("="*60)